# 🚀 RAG Agent Evaluation with RAGAS Framework

## Evaluating Your AI Agent's Quality! 🤖

This notebook demonstrates how to **evaluate a Retrieval-Augmented Generation (RAG) agent** using the **RAGAS** (Retrieval Augmented Generation Assessment) framework. You'll learn how to measure whether your agent retrieves the right information and generates faithful, relevant answers.

### 🎯 What You'll Learn

By the end of this lab, you'll master:

1. **🔧 Environment Setup** - Install and configure RAGAS evaluation dependencies
2. **🧠 RAG Pipeline Construction** - Build a complete RAG agent with LangGraph, ChromaDB, and tools
3. **📊 RAGAS Metrics** - Understand faithfulness, answer relevancy, context precision, and context recall
4. **🧪 Evaluation Execution** - Run end-to-end evaluation on your RAG agent with ground truth comparisons
5. **📈 Results Analysis** - Interpret evaluation scores and identify areas for improvement

### 🌟 Why Evaluate RAG Agents?

Building a RAG agent is only half the battle — you need to know if it actually works well:
- **🔍 Faithfulness**: Does the answer stay true to the retrieved context, or does it hallucinate?
- **🎯 Answer Relevancy**: Does the generated answer actually address the user's question?
- **📋 Context Precision**: Are the retrieved documents actually relevant to the query?
- **📦 Context Recall**: Did the retriever find all the relevant documents?

### 📋 Prerequisites

- **Python 3.8+** installed on your system
- **Azure OpenAI access** (API key and endpoint)
- **Basic understanding of RAG pipelines** (retrieval + generation)
- **Familiarity with LangGraph** from previous sessions

---

*💡 **Pro Tip**: RAGAS provides a standardized way to evaluate RAG systems — think of it as unit tests for your AI agent's knowledge retrieval and answer generation!*

## 1. 🛠️ Environment Setup & Installation

### 📦 Package Installation

First, let's install all the required packages for our RAG evaluation journey:

- **`langgraph`**: Framework for building stateful agent workflows
- **`langchain` / `langchain-community` / `langchain-openai`**: Core LLM application libraries and integrations
- **`chromadb`**: Vector database for document storage and retrieval
- **`ragas`**: The RAGAS evaluation framework — our star of the show!
- **`datasets`**: Hugging Face datasets library, used by RAGAS for data handling
- **`pysqlite3-binary`**: SQLite3 replacement needed by ChromaDB on some platforms

### 💡 Why These Packages?
The evaluation pipeline requires both the **RAG agent** (LangGraph + ChromaDB) and the **evaluation framework** (RAGAS + datasets). We install everything upfront to avoid dependency issues later.

In [1]:
# 📦 Install all required packages for RAG agent + RAGAS evaluation
# -U flag ensures we get the latest versions of each package
!pip install -U langgraph langchain langchain-community==0.4.0 langchain-openai chromadb doppler-sdk pysqlite3-binary ragas datasets --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### 🔧 SQLite3 Compatibility Patch

ChromaDB requires a modern version of SQLite3. Some environments ship with an older version, so we swap in `pysqlite3` as a drop-in replacement. This cell **must run before** any ChromaDB imports.

In [2]:
import sys

# 🔧 Attempt to import pysqlite3 and register it as the system's sqlite3 module
# This is required because ChromaDB needs SQLite >= 3.35.0
try:
    import pysqlite3 as sqlite3
    sys.modules["sqlite3"] = sqlite3  # Override the built-in sqlite3 with pysqlite3
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "pysqlite3-binary is required. Run the first cell to install it."
    ) from exc

print("✅ SQLite3 compatibility patch applied successfully!")

✅ SQLite3 compatibility patch applied successfully!


## 2. 🔐 Azure OpenAI Configuration

### Setting Up Your AI Brain 🧠

This section configures the connection to Azure OpenAI, which powers both our RAG agent and the RAGAS evaluation framework.

### 🔧 Configuration Steps:
1. **Load Credentials** - Import API keys and endpoints securely via `init_creds`
2. **Initialize Chat Model** - Set up `gpt-4o-mini` for reasoning and answer generation
3. **Initialize Embeddings** - Set up `text-embedding-3-small` for vector similarity search
4. **Validation** - Ensure all credentials are present before proceeding

### 💡 Models We're Using:
- **`gpt-4o-mini`**: Fast, cost-efficient model for the agent's reasoning and RAGAS evaluation
- **`text-embedding-3-small`**: Creates vector representations for document retrieval and RAGAS embedding-based metrics

### 🔑 Key Concept: Dual-Purpose Models
The same LLM and embeddings model serve **two roles** in this notebook:
1. **Agent role**: Power the RAG pipeline (routing, retrieval, synthesis)
2. **Evaluator role**: RAGAS uses them to compute metrics like faithfulness and relevancy

In [3]:
# 📚 Import credential management and LangChain Azure OpenAI classes
import init_creds as creds
import sys
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

# 🔗 Add parent directory to path so shared modules (like init_creds) are accessible
sys.path.insert(1, '../../')

# 🔑 Load Azure OpenAI credentials securely from the creds module
AZURE_OPENAI_KEY = creds.get_api_key()             # Your Azure OpenAI API key
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()        # Your Azure OpenAI endpoint URL
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"     # API version for Azure OpenAI

# ⚠️ Validate that credentials are present — fail fast if missing
if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

# 🧠 Initialize the Chat LLM — used for agent reasoning AND RAGAS evaluation
llm = AzureChatOpenAI(
    model="gpt-4o-mini",                           # Fast, efficient model for conversations
    azure_endpoint=AZURE_OPENAI_ENDPOINT,           # Azure endpoint URL
    api_key=AZURE_OPENAI_KEY,                       # API authentication key
    api_version=AZURE_OPENAI_API_VERSION            # API version string
)

# 🔍 Initialize the Embeddings model — used for vector search AND RAGAS metrics
embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",                 # Compact, fast embedding model
    api_key=AZURE_OPENAI_KEY,                       # Same API key as the chat model
    azure_endpoint=AZURE_OPENAI_ENDPOINT,           # Same endpoint
    api_version=AZURE_OPENAI_API_VERSION            # Same API version
)

print("✅ LLM and Embeddings initialized successfully!")
print(f"🧠 Language Model: gpt-4o-mini")
print(f"🔍 Embeddings Model: text-embedding-3-small")

✅ LLM and Embeddings initialized successfully!
🧠 Language Model: gpt-4o-mini
🔍 Embeddings Model: text-embedding-3-small


In [4]:
# 💡 ALTERNATIVE: If you're using OpenAI directly (not Azure), uncomment these lines
# and comment out the Azure configuration above.
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 3. 📥 Download the Knowledge Base

### Preparing Our RAG Data Source 📄

Every RAG system needs a **knowledge base** — a collection of documents that the retriever searches through. Here we download a sample health manual that our agent will use to answer questions about fitness, heart rate, steps, and sleep.

### 🔑 Key Concept: Why a Knowledge Base Matters for Evaluation
- The quality of your **retrieval** directly impacts your **evaluation scores**
- RAGAS metrics like **context precision** and **context recall** measure how well the retriever finds relevant passages from this knowledge base
- A well-structured knowledge base with clear, factual content will yield better evaluation results

In [5]:
# 📥 Download the health manual text file from S3 — this is our RAG knowledge base
# The file contains health/fitness information that the agent will retrieve from
!curl -o manual.txt "https://confident-bucket.s3.us-east-1.amazonaws.com/manual.txt"

print("✅ Knowledge base downloaded: manual.txt")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 18482  100 18482    0     0  59452      0 --:--:-- --:--:-- --:--:-- 59619
✅ Knowledge base downloaded: manual.txt


## 4. 🏗️ Building the RAG Agent Pipeline

### Constructing a Full Agent with Routing, Retrieval, Tools & Synthesis 🔀

This is the core of our system — a complete **LangGraph-based RAG agent** with multiple capabilities. The agent uses an intelligent router to decide how to handle each query.

### 🧠 Architecture Overview:

```
User Query → Router Node → ┬→ Retrieval Node → Synthesis Node → Answer
                           ├→ Tool Execution Node → Synthesis Node → Answer
                           └→ Direct Synthesis Node → Answer
```

### 🔑 Key Components:

| Component | Purpose |
|---|---|
| **AgentState** | TypedDict that tracks messages, query, tools, context, and routing decisions |
| **Vector Store (ChromaDB)** | Stores document embeddings for semantic retrieval |
| **Tools** | Simulated health data tools (steps, heart rate, sleep duration) |
| **Router Node** | LLM-powered decision maker — routes to retrieval, tools, or direct answer |
| **Retrieval Node** | Fetches relevant documents from ChromaDB |
| **Tool Execution Node** | Runs selected tools and collects outputs |
| **Synthesis Node** | Combines all context sources into a final, coherent answer |

### 💡 Why This Matters for Evaluation:
- The **routing decisions** affect which path the query takes — and which contexts get captured
- RAGAS evaluates the **end-to-end pipeline**, not just individual components
- Understanding the architecture helps you interpret evaluation scores correctly

In [6]:
import os
import random
from typing import Annotated, List, TypedDict, Literal
from chromadb.config import Settings
from langchain_community.vectorstores import Chroma
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

# ============================================================
# 📋 STEP 1: Define the Agent State Schema
# ============================================================
# The state is a TypedDict that flows through every node in the graph.
# Each node can read from and write to this shared state.

class AgentState(TypedDict):
    """State schema for the RAG agent — tracks all information as it flows through the graph"""

    messages: Annotated[List[BaseMessage], add_messages]  # Conversation history (auto-appended)
    query: str                                             # The user's original question
    selected_tools: List[str]                              # Tools chosen by the router
    retrieved_context: str                                 # Text retrieved from the vector store
    tool_outputs: List[str]                                # Results from tool execution
    next_action: str                                       # Routing decision: "retrieval", "tools", or "direct"


# ============================================================
# 📚 STEP 2: Set Up the Vector Store (ChromaDB)
# ============================================================
# This function loads our knowledge base, splits it into chunks,
# embeds each chunk, and stores them in ChromaDB for fast retrieval.

def setup_vector_store():
    """Set up the vector database with documents from the downloaded manual.txt"""
    text_file_path = "manual.txt"  # Path to our downloaded knowledge base

    try:
        # 📄 Load the text file as a LangChain Document object
        loader = TextLoader(text_file_path, encoding="utf-8")
        docs = loader.load()

        # ✂️ Split the document into smaller chunks for better retrieval
        # chunk_size=500: Each chunk is ~500 characters
        # chunk_overlap=50: Adjacent chunks share 50 chars for context continuity
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=50
        )
        doc_splits = text_splitter.split_documents(docs)

        # 🔢 Embed chunks and store in ChromaDB vector database
        # Uses the Azure OpenAI embeddings configured in Section 2
        vector_store = Chroma.from_documents(doc_splits, embeddings)

        # 🔍 Return as a retriever interface for easy querying
        return vector_store.as_retriever()

    except FileNotFoundError:
        print(f"❌ Error: File '{text_file_path}' not found. Please run the download cell first.")
        return None
    except Exception as e:
        print(f"❌ Error loading document: {str(e)}")
        return None

# 🚀 Initialize the retriever — this loads and indexes the knowledge base
retriever = setup_vector_store()
print("✅ Vector store initialized!" if retriever else "⚠️ Vector store setup failed")


# ============================================================
# 🛠️ STEP 3: Define Agent Tools
# ============================================================
# These are simulated health data tools. In a real system, these would
# connect to databases, APIs, or sensors. For evaluation purposes,
# they return random values within realistic ranges.

@tool
def get_last_day_steps():
    """Get the last day's steps from the database"""
    return random.randint(1000, 5000)  # 🚶 Simulated step count


@tool
def get_last_day_average_heart_rate():
    """Get the last day's average heart rate from the database"""
    return random.randint(60, 100)  # ❤️ Simulated BPM


@tool
def get_last_day_average_sleep_duration_in_hours():
    """Get the last day's average sleep duration from the database"""
    return random.randint(3, 10)  # 😴 Simulated hours of sleep


# 📦 Tool registry — maps tool names to tool functions for dynamic lookup
tools = [
    get_last_day_steps,
    get_last_day_average_heart_rate,
    get_last_day_average_sleep_duration_in_hours,
]
tool_registry = {tool.name: tool for tool in tools}
print(f"✅ {len(tools)} tools registered: {[t.name for t in tools]}")


# ============================================================
# 🔀 STEP 4: Define the Router Node
# ============================================================
# The router uses the LLM with structured output to decide
# which path the query should take: retrieval, tools, or direct.

class RouteQuery(BaseModel):
    """Schema for routing decisions — enforces structured LLM output"""

    reasoning: str = Field(description="Reasoning for the routing decision")
    route: Literal["retrieval", "tools", "direct"] = Field(
        description="Where to route the query"
    )
    tools_needed: List[str] = Field(
        description="List of tools needed if route is 'tools'"
    )
    retrieval_query: str = Field(
        description="Optimized query for retrieval if route is 'retrieval'"
    )


def router_node(state: AgentState) -> AgentState:
    """Route the query to the appropriate processing path using LLM reasoning"""

    # 📝 System prompt tells the LLM about available routes and tools
    system_prompt = """You are an intelligent router that decides how to process user queries.
    
    Available options:
    - 'retrieval': Query needs information from the knowledge base
    - 'tools': Query needs external tools (web search, calculations, etc.)  
    - 'direct': Query can be answered directly with general knowledge
    
    Available tools: {tools}
    
    Analyze the user query and decide the best routing approach. If tools are needed,
    specify which ones. If retrieval is needed, optimize the query for better results."""

    user_query = state["messages"][-1].content  # Extract the latest user message

    # 🧠 Use structured output to force the LLM to return a RouteQuery object
    structured_llm = llm.with_structured_output(
        RouteQuery, method="function_calling"
    )

    # 🔄 Invoke the LLM to make the routing decision
    response = structured_llm.invoke(
        [
            {
                "role": "system",
                "content": system_prompt.format(tools=[t.name for t in tools]),
            },
            {"role": "user", "content": user_query},
        ]
    )

    # 📤 Return the routing decision as state updates
    return {
        "query": user_query,
        "next_action": response.route,                  # Which path to take
        "selected_tools": response.tools_needed,         # Tools to execute (if any)
        "retrieved_context": (
            response.retrieval_query if response.route == "retrieval" else ""
        ),
    }


# ============================================================
# ⚙️ STEP 5: Define Processing Nodes
# ============================================================

def tool_execution_node(state: AgentState) -> AgentState:
    """Execute selected tools and collect their outputs"""
    tool_outputs = []
    for tool_name in state["selected_tools"]:
        if tool_name in tool_registry:
            tool = tool_registry[tool_name]
            try:
                # 🔧 Invoke each tool with the original query
                output = tool.invoke({"query": state["query"]})
                tool_outputs.append(f"{tool_name}: {output}")
            except Exception as e:
                tool_outputs.append(f"{tool_name}: Error - {str(e)}")

    return {"tool_outputs": tool_outputs}


def retrieval_node(state: AgentState) -> AgentState:
    """Execute retrieval from the ChromaDB vector database"""
    # Use the optimized retrieval query if available, otherwise the raw query
    query = state["retrieved_context"] or state["query"]

    try:
        # 🔍 Retrieve the most relevant document chunks from the vector store
        docs = retriever.invoke(query)
        # 📄 Join all retrieved passages into a single context string
        context = "\n\n".join([doc.page_content for doc in docs])
        return {"retrieved_context": context}
    except Exception as e:
        return {"retrieved_context": f"Retrieval error: {str(e)}"}


def response_synthesis_node(state: AgentState) -> AgentState:
    """Synthesize a final response from all available information sources"""

    # 📋 Gather context from retrieval and/or tool outputs
    context_parts = []
    if state.get("retrieved_context"):
        context_parts.append(f"Knowledge Base Context:\n{state['retrieved_context']}")
    if state.get("tool_outputs"):
        tool_context = "\n".join(state["tool_outputs"])
        context_parts.append(f"Tool Outputs:\n{tool_context}")

    context = "\n\n".join(context_parts)

    # 🧠 Prompt the LLM to synthesize a comprehensive answer
    system_prompt = """You are a helpful assistant that synthesizes information from multiple sources.
    
    Use the provided context to answer the user's question accurately and comprehensively.
    If using information from the context, be sure to reference it appropriately.
    If the context doesn't contain enough information, acknowledge this limitation."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Question: {state['query']}\n\nContext:\n{context}"},
    ]

    # 🤖 Generate the final answer
    response = llm.invoke(messages)
    return {"messages": [AIMessage(content=response.content)]}


# ============================================================
# 🔀 STEP 6: Define Routing Logic & Build the Graph
# ============================================================

def intial_route_decision(state: AgentState) -> str:
    """Determine which node to visit next based on the router's decision"""
    next_action = state.get("next_action", "direct")

    if next_action == "tools":
        return "tools"          # → Tool Execution Node
    if next_action == "retrieval":
        return "retrieval"      # → Retrieval Node
    return "retrieval"          # Default fallback → Retrieval Node


def create_rag_graph():
    """Create and compile the complete RAG workflow graph"""

    # 🏗️ Initialize the state graph with our AgentState schema
    workflow = StateGraph(AgentState)

    # 📦 Add all processing nodes to the graph
    workflow.add_node("router", router_node)              # Decision-making node
    workflow.add_node("retrieval", retrieval_node)         # Document retrieval node
    workflow.add_node("tools", tool_execution_node)        # Tool execution node
    workflow.add_node("synthesis", response_synthesis_node) # Answer synthesis node

    # 🔗 Define the graph edges (flow of execution)
    workflow.add_edge(START, "router")                     # Entry point → Router
    workflow.add_conditional_edges(                         # Router → conditional branching
        "router",
        intial_route_decision,
        {
            "retrieval": "retrieval",                      # If retrieval → Retrieval Node
            "tools": "tools",                              # If tools → Tool Node
            "synthesis": "synthesis",                      # If direct → Synthesis Node
        },
    )
    workflow.add_edge("retrieval", "synthesis")            # Retrieval → Synthesis
    workflow.add_edge("tools", "synthesis")                # Tools → Synthesis
    workflow.add_edge("synthesis", END)                    # Synthesis → Done

    return workflow.compile()


# 🚀 Create and compile the RAG agent graph
app = create_rag_graph()
print("✅ RAG agent graph compiled successfully!")


✅ Vector store initialized!
✅ 3 tools registered: ['get_last_day_steps', 'get_last_day_average_heart_rate', 'get_last_day_average_sleep_duration_in_hours']
✅ RAG agent graph compiled successfully!


## 5. 🧪 Query Helper Functions

### Running Queries & Capturing Context for Evaluation 📤

Before we can evaluate our RAG agent, we need helper functions that:
1. **Run queries** through the LangGraph pipeline
2. **Capture the retrieved contexts** — RAGAS needs these to compute context-based metrics
3. **Extract the final answer** — RAGAS compares this against ground truth

### 🔑 Key Concept: Why We Need `run_rag_query_with_context`
RAGAS requires **four pieces of data** for each evaluation sample:
- **question**: The user's query
- **answer**: The agent's generated response
- **contexts**: The retrieved document chunks (list of strings)
- **ground_truth**: The expected correct answer

The `run_rag_query_with_context` function captures both the answer AND the retrieved contexts in a single pass through the agent pipeline.

In [7]:
# 📋 Define the initial (empty) state template for the RAG agent
initial_state = {
    "query": "",
    "selected_tools": [],
    "retrieved_context": "",
    "tool_outputs": [],
    "next_action": "",
}


def _invoke_graph(query: str) -> dict:
    """Helper to invoke the workflow with a fresh state for each query."""
    # 🔄 Create a fresh copy of the initial state to avoid mutation between runs
    state = initial_state.copy()
    state["messages"] = [HumanMessage(content=query)]  # Add the user query as a message
    state["query"] = query                              # Store the raw query string
    return app.invoke(state)                            # Run the full LangGraph pipeline


def run_rag_query(query: str) -> str:
    """Run a query through the RAG system and return only the final answer."""
    result = _invoke_graph(query)
    final_message = result["messages"][-1]  # The last message is the agent's answer
    return final_message.content


def run_rag_query_with_context(query: str) -> tuple[str, list[str]]:
    """
    Run a query and capture BOTH the answer AND retrieved contexts.
    This is essential for RAGAS evaluation — it needs contexts to compute
    faithfulness, context precision, and context recall metrics.
    """
    result = _invoke_graph(query)
    final_message = result["messages"][-1].content  # Extract the answer text

    # 📚 Attempt to capture the retrieved contexts from the graph state
    contexts: list[str] = []
    graph_context = result.get("retrieved_context")

    if graph_context:
        # ✅ Context was carried through the graph state
        contexts.append(graph_context)
    else:
        # 🔄 Fallback: Re-run retrieval directly if context wasn't captured in state
        try:
            docs = retriever.invoke(result.get("query", query))
            contexts = [doc.page_content for doc in docs]
        except Exception as exc:
            contexts = [f"Retrieval error: {exc}"]

    return final_message, contexts


# 🧪 Test the query helper with a sample question
print("🔍 Testing query helper...")
answer, contexts = run_rag_query_with_context("What is the average heart rate of the user?")
print(f"✅ Answer received ({len(answer)} chars)")
print(f"📚 Contexts captured: {len(contexts)} passages")

🔍 Testing query helper...
✅ Answer received (58 chars)
📚 Contexts captured: 4 passages


## 6. 📊 RAGAS Metrics Configuration

### Setting Up the Evaluation Framework 🎯

Now we configure the **RAGAS evaluation metrics** — the core of this lab. RAGAS provides four key metrics that together give a comprehensive view of RAG system quality.

### 🔑 The Four RAGAS Metrics:

| Metric | What It Measures | Score Range |
|---|---|---|
| **🔍 Faithfulness** | Does the answer only use information from the retrieved context? Catches hallucinations. | 0.0 - 1.0 |
| **🎯 Answer Relevancy** | Does the answer actually address the user's question? | 0.0 - 1.0 |
| **📋 Context Precision** | Are the retrieved documents relevant to the question? (Precision = relevant / total retrieved) | 0.0 - 1.0 |
| **📦 Context Recall** | Did the retriever find all the relevant documents? (Recall = retrieved relevant / total relevant) | 0.0 - 1.0 |

### 💡 How RAGAS Computes These Metrics:
- **Faithfulness**: The LLM extracts claims from the answer, then checks if each claim is supported by the context
- **Answer Relevancy**: The LLM generates hypothetical questions from the answer, then measures similarity to the original question
- **Context Precision**: The LLM judges whether each retrieved passage is relevant to the question
- **Context Recall**: The LLM checks whether the ground truth answer can be attributed to the retrieved contexts

### ⚠️ Important Note:
We inject our **Azure OpenAI models** into each RAGAS metric so it uses our models instead of defaulting to OpenAI directly.

In [8]:
# 📊 Import RAGAS evaluation components
from datasets import Dataset          # Hugging Face Dataset — RAGAS expects data in this format
from ragas import evaluate            # The main RAGAS evaluation function
from ragas.metrics import (
    answer_relevancy,                 # 🎯 Does the answer address the question?
    context_precision,                # 📋 Are retrieved docs relevant to the query?
    context_recall,                   # 📦 Did we retrieve all relevant docs?
    faithfulness,                     # 🔍 Is the answer faithful to the context (no hallucinations)?
)

# 🔧 Point RAGAS metrics at our Azure OpenAI models
# By default, RAGAS tries to use OpenAI directly — we override with our Azure instances
ragas_llm = llm                       # Use our Azure ChatOpenAI for LLM-based evaluation
ragas_embeddings = embeddings          # Use our Azure embeddings for similarity-based evaluation

# 📋 Define the metrics we want to compute
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

# 🔗 Inject our Azure models into each metric that supports them
for metric in metrics:
    if hasattr(metric, "llm"):
        metric.llm = ragas_llm                # Override the LLM used by this metric
    if hasattr(metric, "embeddings"):
        metric.embeddings = ragas_embeddings   # Override the embeddings used by this metric

print("✅ RAGAS metrics configured with Azure OpenAI models!")
print(f"📊 Metrics to evaluate: {[m.name for m in metrics]}")

✅ RAGAS metrics configured with Azure OpenAI models!
📊 Metrics to evaluate: ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']


/tmp/ipykernel_7700/3133438928.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_7700/3133438928.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_7700/3133438928.py:4: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_7700/3133438928.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be remov

## 7. 🧪 Running the RAGAS Evaluation

### Building the Evaluation Dataset & Computing Scores 🏃

Now comes the exciting part — we'll run our RAG agent on sample questions, capture the results, and let RAGAS score the performance!

### 📋 Evaluation Workflow:
1. **Define evaluation samples** — questions paired with ground truth answers
2. **Run each question** through the agent pipeline, capturing answers and contexts
3. **Build a RAGAS dataset** — organize results into the format RAGAS expects
4. **Compute metrics** — RAGAS uses the LLM to judge faithfulness, relevancy, precision, and recall

### 🔑 Key Concept: Ground Truth
- **Ground truth** is the "correct" answer that you provide manually
- RAGAS compares the agent's output against this ground truth to measure **context recall**
- The better your ground truth, the more meaningful your evaluation scores

### 💡 Interpreting Scores:
- **0.0 - 0.3**: Poor performance — significant issues to address
- **0.3 - 0.6**: Moderate — some improvements needed
- **0.6 - 0.8**: Good — system is working reasonably well
- **0.8 - 1.0**: Excellent — high-quality retrieval and generation

In [9]:
# 📝 Define evaluation samples — each has a question and the expected "ground truth" answer
# Ground truth is used by RAGAS to measure context recall (can the context support this answer?)
evaluation_samples = [
    {
        "question": "What is the average heart rate of the user?",
        "ground_truth": "The health manual states the user's average heart rate typically ranges between 60 and 100 beats per minute.",
    },
    {
        "question": "How many steps did the user walk yesterday?",
        "ground_truth": "The manual logs daily steps between 1,000 and 5,000 for the tracked user.",
    },
]


def build_ragas_dataset(samples: list[dict]) -> Dataset:
    """
    Run the agent for each evaluation sample and build a RAGAS-compatible Dataset.
    
    For each sample, this function:
    1. Sends the question through the full RAG agent pipeline
    2. Captures the generated answer and retrieved contexts
    3. Packages everything into the format RAGAS expects
    """
    rows = []
    for sample in samples:
        print(f"🔄 Processing: {sample['question'][:50]}...")
        
        # 🚀 Run the question through our RAG agent, capturing answer + contexts
        answer, contexts = run_rag_query_with_context(sample["question"])
        
        # 📦 Package into RAGAS format: question, answer, contexts, ground_truth
        rows.append(
            {
                "question": sample["question"],                          # The user's query
                "answer": answer,                                        # Agent's generated answer
                "contexts": contexts or ["No retrieval context captured."],  # Retrieved passages
                "ground_truth": sample["ground_truth"],                  # Expected correct answer
            }
        )

    return Dataset.from_list(rows)  # Convert to Hugging Face Dataset format


# 🏗️ Build the evaluation dataset by running all sample questions through the agent
print("📊 Building RAGAS evaluation dataset...")
ragas_dataset = build_ragas_dataset(evaluation_samples)

# 🧪 Run RAGAS evaluation — this uses the LLM to compute all four metrics
print("\n🔬 Running RAGAS evaluation (this may take a moment)...")
ragas_scores = evaluate(
    ragas_dataset,                    # The dataset with questions, answers, contexts, ground truth
    metrics=metrics,                  # Our four metrics: faithfulness, relevancy, precision, recall
    llm=ragas_llm,                    # Azure OpenAI LLM for evaluation judgments
    embeddings=ragas_embeddings,      # Azure OpenAI embeddings for similarity computations
)

# 📊 Display the aggregated scores
print("\n✅ RAGAS Evaluation Complete!")
ragas_scores

📊 Building RAGAS evaluation dataset...
🔄 Processing: What is the average heart rate of the user?...
🔄 Processing: How many steps did the user walk yesterday?...

🔬 Running RAGAS evaluation (this may take a moment)...


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

/home/codespace/.local/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:549: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")



✅ RAGAS Evaluation Complete!


{'faithfulness': 0.0000, 'answer_relevancy': 1.0000, 'context_precision': 0.5000, 'context_recall': 0.5000}

## 8. 📈 Detailed Results Inspection

### Examining Individual Evaluation Samples 🔎

The aggregated scores above give us the big picture, but it's equally important to inspect **each individual sample** to understand where the agent performs well and where it struggles.

### 💡 What to Look For:
- **Answer vs. Ground Truth**: Does the agent's answer align with the expected answer?
- **Contexts**: What passages were actually retrieved? Are they relevant?
- **Mismatches**: If scores are low, inspecting individual samples reveals the root cause

In [10]:
# 🔎 Inspect each evaluation sample in detail
# This shows the question, the agent's answer, the ground truth, and the retrieved contexts
print("=" * 60)
print("📋 DETAILED EVALUATION RESULTS")
print("=" * 60)

for i in ragas_dataset:
    print(f"\n❓ Question: {i['question']}")
    print(f"🤖 Answer: {i['answer']}")
    print(f"✅ Ground Truth: {i['ground_truth']}")
    print(f"📚 Contexts: {i['contexts']}")
    print("-" * 50)

📋 DETAILED EVALUATION RESULTS

❓ Question: What is the average heart rate of the user?
🤖 Answer: The average heart rate of the user is 74 beats per minute.
✅ Ground Truth: The health manual states the user's average heart rate typically ranges between 60 and 100 beats per minute.
📚 Contexts: ["Your resting heart rate, the number of times your heart beats per minute while at rest, is a key indicator of your cardiovascular fitness. A lower resting heart rate generally implies more efficient heart function.[10]\nWhat's a Healthy Range?\nFor most adults, a normal resting heart rate is between 60 and 100 beats per minute (bpm).[10][11][12] Well-trained athletes may have a resting heart rate closer to 40 bpm.[10]\nBest Practices for a Healthy Heart Rate:", 'Understanding Your Resting Heart Rate (RHR)\nYour RHR is the number of times your heart beats per minute when you are completely at rest.[10] For most adults, a normal RHR falls between 60 and 100 beats per minute (bpm).[11][12] A lower R

## 9. ✅ Summary & Key Takeaways

### What We Accomplished 🎉

In this lab, we built a **complete RAG evaluation pipeline** from scratch:

1. **🏗️ Built a RAG Agent** — A LangGraph-based agent with routing, retrieval (ChromaDB), tool execution, and answer synthesis
2. **📊 Configured RAGAS Metrics** — Set up faithfulness, answer relevancy, context precision, and context recall
3. **🧪 Ran End-to-End Evaluation** — Executed evaluation samples through the agent and computed standardized scores
4. **🔎 Inspected Results** — Examined individual samples to understand agent behavior

### 🧠 Key Concepts to Remember:

| Concept | Why It Matters |
|---|---|
| **Faithfulness** | Prevents hallucination — the #1 concern in production RAG systems |
| **Answer Relevancy** | Ensures the agent actually answers what was asked |
| **Context Precision** | Measures retrieval quality — are we finding the right documents? |
| **Context Recall** | Measures retrieval completeness — are we finding ALL relevant documents? |
| **Ground Truth** | Your evaluation is only as good as your ground truth annotations |

### 🚀 Next Steps:
- **Add more evaluation samples** to get statistically meaningful results
- **Experiment with chunk sizes** and see how they affect context precision/recall
- **Compare different models** — swap gpt-4o-mini for gpt-4o and compare scores
- **Integrate into CI/CD** — run RAGAS evaluations automatically on every pipeline change
- **Explore other RAGAS metrics** — answer correctness, answer similarity, and more

---

*💡 **Pro Tip**: In production, aim for faithfulness > 0.8 and context precision > 0.7 as minimum thresholds. Low faithfulness means your agent is hallucinating!*